In [ ]:
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoTokenizer

# nbconvert runs from notebooks/, so find the repo root
repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())

corpus_dir = repo_root / "data/corpus/habsburg_clean"
output_path = repo_root / "data/tokenized/habsburg"
cache_dir = repo_root / "data/tokenized/cache"  # ~200GB of intermediates, safe to delete after
tokenizer_path = repo_root / "data/tokenizer/habsburg-48k"

sequence_length = 4096   # must match scripts/train.py
val_documents = 1000     # ~20M tokens
seed = 42
num_workers = 12
document_batch = 100     # documents per map batch; caps peak memory during packing


In [ ]:
corpus_files = sorted(str(p) for p in corpus_dir.glob("*.parquet"))
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
print(f"{len(corpus_files)} year files, vocab {len(tokenizer):,}")

dataset = load_dataset(
    "parquet", data_files=corpus_files, split="train",
    columns=["text"], cache_dir=str(cache_dir),
)
splits = dataset.train_test_split(test_size=val_documents, seed=seed)
print(splits)


In [ ]:
def tokenize(batch):
    # append EOS so group_texts keeps document boundaries
    input_ids = tokenizer(batch["text"], verbose=False)["input_ids"]
    for ids in input_ids:
        ids.append(tokenizer.eos_token_id)
    return {"input_ids": input_ids}


tokenized = splits.map(
    tokenize,
    batched=True,
    batch_size=document_batch,
    remove_columns=["text"],
    num_proc=num_workers,
    desc="Tokenizing",
)
tokenized


In [ ]:
def group_texts(batch):
    concatenated = sum(batch["input_ids"], [])
    total = (len(concatenated) // sequence_length) * sequence_length
    return {"input_ids": [concatenated[i : i + sequence_length]
                          for i in range(0, total, sequence_length)]}


chunked = tokenized.map(
    group_texts,
    batched=True,
    batch_size=document_batch,
    num_proc=num_workers,
    desc="Packing",
)
chunked


In [ ]:
shutil.rmtree(output_path, ignore_errors=True)
output_path.mkdir(parents=True, exist_ok=True)
chunked.save_to_disk(str(output_path))

train_sequences = len(chunked["train"])
val_sequences = len(chunked["test"])
print(f"Saved to {output_path}")
print(f"Train: {train_sequences:,} sequences ({train_sequences * sequence_length / 1e9:.2f}B tokens)")
print(f"Val:   {val_sequences:,} sequences ({val_sequences * sequence_length / 1e6:.0f}M tokens)")
print(f"5 epochs = {5 * train_sequences * sequence_length / 1e9:.0f}B tokens seen")


In [ ]:
sample_ids = chunked["train"][0]["input_ids"][:60]
print(f"First 60 tokens: {sample_ids}")
print(f"Decoded: {tokenizer.decode(sample_ids)}")
